# Simple RAG Pipeline with LangChain + OpenAI

In this live session, we will build a **Retrieval-Augmented Generation (RAG)** system that can answer questions about the **WEF Financial Policy** document.

**What we will cover:**
1. Load the PDF
2. Split into chunks
3. Create embeddings & vector store
4. Build a RAG chain
5. Ask real questions about the financial policy

In [16]:
# !pip install langchain langchain-openai langchain-classic langchain-community langchain-chroma pypdf -q
import os
from dotenv import load_dotenv

load_dotenv()

print("OpenAI API Key loaded: ", bool(os.getenv("OPENAI_API_KEY")))


OpenAI API Key loaded:  True


## Load the PDF Document

In [17]:
from langchain_community.document_loaders import PyPDFLoader

# Load the WEF Financial Policy PDF
loader = PyPDFLoader("/content/wef-financial-policy.pdf")
docs = loader.load()

print(f"\nFirst page preview:\n{docs[0].page_content[:800]}...")


First page preview:
Financial Policy  
Policy Name WEF Financial Policy 
Policy Category Business 
Policy Number 2002-001 
Policy Origination and 
Review Dates 
July 2002; July 2005; April 2007; July 2008; July 2009; 
November 2010; October 2013; April 2014; July 2014; 
February 2015; January 2016; April 2017; September 2017; 
April 2018; July 2018; April 2019; February 2020; April 2021; 
August 2024 
Requirements No legal requirements  
Review Cycle  3 Years 
Legal Review Required No 
A. General Policy Statement  
The Board of Trustees requires that the finances of WEF be managed in a reasonable and prudent 
manner and that spending be consistent with the budget and the strategic interests of WEF.   
Contingent issues arise between budget meetings which can have an unplanned impact on the current 
and future...


## Split Documents into Chunks

In [21]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 150,
    length_function = len,
    separators = ["\n\n", "\n", " ", ""]
)

splits = text_splitter.split_documents(docs)

print(f"\nSample chunk (first 400 chars): \n{splits[8].page_content[:400]}...")



Sample chunk (first 400 chars): 
completed IRS tax return form 990 (in either printed or electronic format) to the Board of Trustees and 
to organizations entitled to receive a copy because of contractual agreement.  
 
Authority to distribute the statements to other individuals or firms requesting them will be at the 
discretion of the Executive Director. (REV 04/17) 
 
AUDITS: THE ENGAGEMENT LETTER 
It is the policy of WEF that...


## Create Embeddings & Vector Store (Chroma)

In [22]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents = splits,
    embedding = embeddings,
    collection_name = "wef_financial_policy_rag"
)

print("Vector store create successfully!")



Vector store create successfully!


## Create Retrieve

In [25]:
retriever = vectorstore.as_retriever(
    search_type = 'similarity',
    search_kwargs={'k': 5} # Retrieve top 5 most relevant chunks
)

# Quick test
test_query = "What is the capitalization cutoff point?"
retrieved = retriever.invoke(test_query)
print(f"Retrieved {len(retrieved)} chunks for test query.")
for i, doc in enumerate(retrieved[:5]):
  print(f"\n--- Chunk {i+1} ---")
  print(doc.page_content[:350] + "...")

Retrieved 5 chunks for test query.

--- Chunk 1 ---
CAPITALIZATION CUTOFF POINTS 
It is the policy of this Federation to expense assets in the period purchased if these assets cost $2,500 
or less individually. 
 
Assets costing in excess of $2,500 individually will be capitalized and depreciated in accordance with 
WEF’s depreciation policies. 
 
Repairs and improvements to real property will be ca...

--- Chunk 2 ---
CAPITALIZATION CUTOFF POINTS 
It is the policy of this Federation to expense assets in the period purchased if these assets cost $2,500 
or less individually. 
 
Assets costing in excess of $2,500 individually will be capitalized and depreciated in accordance with 
WEF’s depreciation policies. 
 
Repairs and improvements to real property will be ca...

--- Chunk 3 ---
Executive Director or their designee who will determine appropriateness for use in conformance with 
this policy. (REV 02/15) 
   
NON-SUFFICIENT FUNDS CHECKS 
It is the policy of WEF to include checks retu

## Build the RAG Chain

In [54]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

system_prompt = (
    "You are a helpful assistant specialized in answering questions about the WEF Financial Policy. "
    "Use only the provided context from the policy document to answer questions. "
    "If the answer is not in the context, clearly say you don't have that information. "
    "Be concise and quote policy sections when helpful."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("RAG Chain is ready!")


RAG Chain is ready!


## Ask Questions

In [55]:
def ask_policy_question(question):
  print(f"Question: {question}\n")
  response = rag_chain.invoke({"input": question})

  print("Answer: ")
  print(response["answer"])

  print("\nSource used:")
  for i, doc in enumerate(response['context']):
      page = doc.metadata.get("page", "N/A")
      print(f" [{i+1}] Page {page}")


### Example Q&A

In [56]:
# Question 1
ask_policy_question("What is the capitalization cutoff for assets?")

Question: What is the capitalization cutoff for assets?

Answer: 
The capitalization cutoff for assets is $2,500. Assets costing $2,500 or less individually will be expensed in the period purchased, while assets costing in excess of $2,500 will be capitalized and depreciated according to WEF’s depreciation policies.

Source used:
 [1] Page 4
 [2] Page 4
 [3] Page 8
 [4] Page 8
 [5] Page 9


In [57]:
# Question 2
ask_policy_question("What are the bid requirements for services and capital purchases?")

Question: What are the bid requirements for services and capital purchases?

Answer: 
The bid requirements for services and capital purchases are as follows:

- **Services**: Contracts in excess of $100,000 require three (3) bids.
- **Capital Purchases**: Items in excess of $25,000 require three (3) bids.

This is stated in the policy: "It is the policy of WEF to solicit three (3) bids for the following expenditures: ... Capital Purchases Items in excess of $25,000 ... Services Contracts in excess of $100,000." (REV 04/19)

Source used:
 [1] Page 3
 [2] Page 3
 [3] Page 4
 [4] Page 4
 [5] Page 3
